In [4]:
import numpy as np

In [5]:
class SlaterDeterminant:
    def __init__(self, n_basis: int | None = None, n_electrons_up: int | None = None,  n_electrons_down: int | None = None, initial_phi_up: np.ndarray | None = None, initial_phi_down: np.ndarray | None = None, weight: float = 0):
        """
        Initializes the Slater determinant with N x M_up and N x M_down matrices.
        N is the size of the basis, M is the number of occupied orbitals (electrons).
        """
        self.weight = weight
        if initial_phi_up is not None and initial_phi_down is not None:
            self.phi_up = np.array(initial_phi_up)
            self.phi_down = np.array(initial_phi_down)
            self.n_basis, self.n_electrons_up = self.phi_up.shape
            _, self.n_electrons_down = self.phi_down.shape

        elif n_basis is not None and n_electrons_up is not None and n_electrons_down is not None:
            self.n_basis = n_basis
            self.n_electrons_up = n_electrons_up
            self.n_electrons_down = n_electrons_down
            self.phi_up = np.zeros((n_basis, n_electrons_up))
            self.phi_down = np.zeros((n_basis, n_electrons_down))

        else:
            raise ValueError("Must provide either initial_phi_up and initial_phi_down or n_basis, n_electrons_up and n_electrons_down.")


    def __repr__(self):
        return f"SlaterDeterminant(n_basis={self.n_basis}, n_electrons_up={self.n_electrons_up}, n_electrons_down={self.n_electrons_down})"


    def norm(self):
        """
        Calculates the squared norm <self | self> of the Slater determinant.
        """
        overlap_matrix_up = self.phi_up.conj().T @ self.phi_up
        overlap_matrix_down = self.phi_down.conj().T @ self.phi_down
        return np.linalg.det(overlap_matrix_up) * np.linalg.det(overlap_matrix_down)


    def initialize(self, system, method: str):
        if method == "random":
            self.phi_up = np.random.rand(self.n_basis, self.n_electrons_up)
            self.phi_down = np.random.rand(self.n_basis, self.n_electrons_down)
        elif method == "zero":
            self.phi_up = np.zeros((self.n_basis, self.n_electrons_up))
            self.phi_down = np.zeros((self.n_basis, self.n_electrons_down))
        elif method == "non-interacting":
            evecs = system.get_non_interacting_evecs()
            self.phi_up = evecs[:, :self.n_electrons_up]
            self.phi_down = evecs[:, :self.n_electrons_down]
        else:
            raise ValueError("Invalid initialization method.")



class System:
    def __init__(self, n_basis: int):
        """
        Base class for physical systems.
        """
        self.n_basis = n_basis
        self.h_kin = None
        self.evals = None
        self.evecs = None


    def _build_non_interacting_hamiltonian(self):
        raise NotImplementedError("Subclasses must implement this method.")


    def get_non_interacting_evecs(self):
        """Diagonalizes the kinetic energy matrix and returns eigenvectors."""
        if self.h_kin is None:
            self.h_kin = self._build_non_interacting_hamiltonian()
        self.evals, self.evecs = np.linalg.eigh(self.h_kin)
        return self.evecs



class HubbardSystem1D(System):
    def __init__(self, n_sites: int, t: float = 1.0, U: float = 10.0, pbc: bool = True):
        """
        Initializes the 1D Hubbard model parameters and builds the non-interacting
        Hamiltonian (hopping) matrix.
        """
        super().__init__(n_basis=n_sites)
        self.t = t
        self.U = U
        self.pbc = pbc

        self.h_kin = self._build_non_interacting_hamiltonian()


    def _build_non_interacting_hamiltonian(self):
        """Builds the N x N non-interacting kinetic energy matrix."""
        H0 = np.zeros((self.n_basis, self.n_basis))
        for i in range(self.n_basis):
            # Nearest neighbor hopping
            H0[i, (i + 1) % self.n_basis] = -self.t
            H0[(i + 1) % self.n_basis, i] = -self.t

        # Remove wrap-around hopping if using open boundary conditions
        if not self.pbc and self.n_basis > 2:
            H0[0, self.n_basis - 1] = 0.0
            H0[self.n_basis - 1, 0] = 0.0

        return H0

In [6]:
state = SlaterDeterminant(n_basis=4, n_electrons_up=3, n_electrons_down=2)

print(state.phi_up)

print(state.phi_down)

system = HubbardSystem1D(4, pbc=False)
print(system.get_non_interacting_evecs()[0])
print(system.get_non_interacting_evecs()[1])
print(system.get_non_interacting_evecs()[2])
print(system.get_non_interacting_evecs()[3])

state.initialize(system, "non-interacting")

print(state.phi_up)

print(state.phi_down)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]
[-0.37174803  0.60150096  0.60150096 -0.37174803]
[-0.60150096  0.37174803 -0.37174803  0.60150096]
[-0.60150096 -0.37174803 -0.37174803 -0.60150096]
[-0.37174803 -0.60150096  0.60150096  0.37174803]
[[-0.37174803  0.60150096  0.60150096]
 [-0.60150096  0.37174803 -0.37174803]
 [-0.60150096 -0.37174803 -0.37174803]
 [-0.37174803 -0.60150096  0.60150096]]
[[-0.37174803  0.60150096]
 [-0.60150096  0.37174803]
 [-0.60150096 -0.37174803]
 [-0.37174803 -0.60150096]]


In [7]:
state = SlaterDeterminant(n_basis=4, n_electrons_up=3, n_electrons_down=2)

In [8]:
state.phi_up

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

In [9]:
state.phi_down

array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]])

In [10]:
system = HubbardSystem1D(4)
print(system.get_non_interacting_evecs()[0])
print(system.get_non_interacting_evecs()[1])
print(system.get_non_interacting_evecs()[2])
print(system.get_non_interacting_evecs()[3])

[ 0.5        -0.70710678  0.          0.5       ]
[ 5.00000000e-01 -2.45326947e-17 -7.07106781e-01 -5.00000000e-01]
[0.5        0.70710678 0.         0.5       ]
[ 5.00000000e-01 -2.45326947e-17  7.07106781e-01 -5.00000000e-01]


In [14]:
system.h_kin

array([[ 0., -1.,  0., -1.],
       [-1.,  0., -1.,  0.],
       [ 0., -1.,  0., -1.],
       [-1.,  0., -1.,  0.]])

In [11]:
state.initialize(system, "non-interacting")

In [12]:
state.phi_up

array([[ 5.00000000e-01, -7.07106781e-01,  0.00000000e+00],
       [ 5.00000000e-01, -2.45326947e-17, -7.07106781e-01],
       [ 5.00000000e-01,  7.07106781e-01,  0.00000000e+00],
       [ 5.00000000e-01, -2.45326947e-17,  7.07106781e-01]])

In [13]:
state.phi_down

array([[ 5.00000000e-01, -7.07106781e-01],
       [ 5.00000000e-01, -2.45326947e-17],
       [ 5.00000000e-01,  7.07106781e-01],
       [ 5.00000000e-01, -2.45326947e-17]])

In [15]:
evals, evecs = np.linalg.eigh(system.h_kin)
print(evals)
print(evecs)

[-2.00000000e+00 -4.51028104e-17  0.00000000e+00  2.00000000e+00]
[[ 5.00000000e-01 -7.07106781e-01  0.00000000e+00  5.00000000e-01]
 [ 5.00000000e-01 -2.45326947e-17 -7.07106781e-01 -5.00000000e-01]
 [ 5.00000000e-01  7.07106781e-01  0.00000000e+00  5.00000000e-01]
 [ 5.00000000e-01 -2.45326947e-17  7.07106781e-01 -5.00000000e-01]]
